In [27]:
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = Path("data/public.jsonl")
RESULTS_DIR = Path("results/classical_baselines")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete")
print("DATA_PATH:", DATA_PATH)
print("RESULTS_DIR:", RESULTS_DIR)

Setup complete
DATA_PATH: data/public.jsonl
RESULTS_DIR: results/classical_baselines


## Load data

In [28]:
assert DATA_PATH.exists(), f"Cannot find {DATA_PATH}. Run this notebook from the competition repo root."

with DATA_PATH.open("r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

num_mcq = sum(bool(x.get("options")) for x in data)
num_free = len(data) - num_mcq

print(f"Loaded {len(data)} public examples")
print(f"MCQ: {num_mcq}")
print(f"Free-form: {num_free}")

Loaded 1126 public examples
MCQ: 375
Free-form: 751


In [29]:
def item_to_text(item: dict) -> str:
    """
    Convert one problem into plain text for shallow BoW models.
    This intentionally removes sequence/reasoning structure.
    """
    question = item["question"]

    if item.get("options"):
        labels = [chr(65 + i) for i in range(len(item["options"]))]
        options_text = "\n".join(
            f"{label}. {option}" for label, option in zip(labels, item["options"])
        )
        return question + "\n\nOptions:\n" + options_text

    return question


def answer_to_label(item: dict) -> str:
    """
    MCQ answer -> 'A', 'B', 'C', ...
    Free-form answer -> joined string. Free-form is not used for these classical MCQ baselines.
    """
    ans = item["answer"]

    if item.get("options"):
        return str(ans).strip().upper()

    if isinstance(ans, list):
        return ", ".join(str(x).strip() for x in ans)

    return str(ans).strip()


def label_to_response(label: str) -> str:
    """Return a minimal boxed response, matching the competition answer-extraction style."""
    return f"\\boxed{{{str(label).strip()}}}"


def extract_letter(text: str) -> str:
    """Extract an MCQ letter from a boxed response or fallback to the last capital letter."""
    m = re.search(r"\\boxed\{\s*([A-Za-z])\s*\}", str(text))
    if m:
        return m.group(1).upper()

    matches = re.findall(r"\b([A-Z])\b", str(text).upper())
    return matches[-1] if matches else ""


def score_mcq_response(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == str(gold_letter).strip().upper()


def robust_stratify_labels(labels: pd.Series):
    """
    train_test_split(stratify=...) fails if any class has fewer than 2 examples.
    Use stratification only when safe.
    """
    counts = labels.value_counts()
    return labels if counts.min() >= 2 else None

## Build Dataframe

In [30]:
records = []
for idx, item in enumerate(data):
    records.append({
        "idx": idx,
        "id": item["id"],
        "is_mcq": bool(item.get("options")),
        "text": item_to_text(item),
        "label": answer_to_label(item),
        "item": item,
    })

df = pd.DataFrame(records)
mcq_df = df[df["is_mcq"]].copy()

print("Full dataframe:", df.shape)
print("MCQ dataframe:", mcq_df.shape)
print("MCQ label distribution:")
print(mcq_df["label"].value_counts().sort_index())

df.head()

Full dataframe: (1126, 6)
MCQ dataframe: (375, 6)
MCQ label distribution:
label
A    35
B    33
C    53
D    36
E    40
F    49
G    39
H    29
I    29
J    32
Name: count, dtype: int64


,idx,id,is_mcq,text,label,item
0,0,0,False,Find the sum of the first $325$ positive even ...,325*(1+325),{'question': 'Find the sum of the first $325$ ...
1,1,1,True,$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ...,F,{'question': '$int_{-infty}^{+infty} frac{a^{3...
2,2,2,False,A roasted turkey is taken from an oven when it...,"143.224229233795, 2.32624773420025",{'question': 'A roasted turkey is taken from a...
3,3,3,False,Reduce the fraction ${\frac{25}{40}}$. [ANS],5/8,{'question': 'Reduce the fraction ${\frac{25}{...
4,4,4,True,"Given $u(x, y) = x^3 + 6x^2y - 3xy^2 - 2y^3$, ...",C,"{'question': 'Given $u(x, y) = x^3 + 6x^2y - 3..."


## Train/dev split

In [31]:
stratify = robust_stratify_labels(mcq_df["label"])

mcq_train_df, mcq_dev_df = train_test_split(
    mcq_df,
    test_size=0.2,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=stratify,
)

mcq_train_df = mcq_train_df.reset_index(drop=True)
mcq_dev_df = mcq_dev_df.reset_index(drop=True)

print("MCQ train size:", len(mcq_train_df))
print("MCQ dev size:", len(mcq_dev_df))
print("\nTrain labels:")
print(mcq_train_df["label"].value_counts().sort_index())
print("\nDev labels:")
print(mcq_dev_df["label"].value_counts().sort_index())

MCQ train size: 300
MCQ dev size: 75

Train labels:
label
A    28
B    26
C    43
D    29
E    32
F    39
G    31
H    23
I    23
J    26
Name: count, dtype: int64

Dev labels:
label
A     7
B     7
C    10
D     7
E     8
F    10
G     8
H     6
I     6
J     6
Name: count, dtype: int64


## Evaluation function

In [32]:
def evaluate_mcq_predictions(name: str, dev_df: pd.DataFrame, pred_labels, train_time_sec: float = 0.0):
    """Evaluate MCQ predictions and return a scored dataframe plus one summary row."""
    pred_labels = [str(x).strip().upper() for x in pred_labels]

    scored_df = dev_df[["id", "is_mcq", "label", "text"]].copy()
    scored_df["gold"] = scored_df["label"]
    scored_df["pred_label"] = pred_labels
    scored_df["response"] = scored_df["pred_label"].apply(label_to_response)
    scored_df["correct"] = [
        score_mcq_response(resp, gold)
        for resp, gold in zip(scored_df["response"], scored_df["gold"])
    ]

    correct = int(scored_df["correct"].sum())
    total = int(len(scored_df))
    acc = 100.0 * correct / total if total else 0.0

    summary = {
        "run_name": name,
        "task_scope": "MCQ only",
        "num_dev": total,
        "num_correct": correct,
        "mcq_acc": acc,
        "free_form_acc": "N/A",
        "overall_acc": "N/A",
        "train_time_sec": train_time_sec,
    }

    print("=" * 70)
    print(name)
    print("=" * 70)
    print(f"MCQ accuracy: {correct} / {total} ({acc:.2f}%)")
    print(f"Train time: {train_time_sec:.2f} sec")
    print("=" * 70)

    return scored_df, summary

## Majority-label baseline

In [33]:
start = time.time()

majority_clf = DummyClassifier(strategy="most_frequent")
majority_clf.fit(np.zeros((len(mcq_train_df), 1)), mcq_train_df["label"])
majority_pred = majority_clf.predict(np.zeros((len(mcq_dev_df), 1)))

majority_scored, majority_summary = evaluate_mcq_predictions(
    name="majority_label_baseline_mcq_only",
    dev_df=mcq_dev_df,
    pred_labels=majority_pred,
    train_time_sec=time.time() - start,
)

majority_label_baseline_mcq_only
MCQ accuracy: 10 / 75 (13.33%)
Train time: 0.00 sec


## BoW + Logistic Regression

In [34]:
def train_bow_logistic_regression(train_df: pd.DataFrame, dev_df: pd.DataFrame, max_features: int = 20000):
    start = time.time()

    vectorizer = CountVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        ngram_range=(1, 2),
        max_features=max_features,
        min_df=1,
    )

    X_train = vectorizer.fit_transform(train_df["text"])
    X_dev = vectorizer.transform(dev_df["text"])

    y_train = train_df["label"].astype(str).str.strip().str.upper().values

    clf = LogisticRegression(
        max_iter=2000,
        solver="lbfgs",
        random_state=RANDOM_SEED,
    )

    print("Training BoW + Logistic Regression...")
    clf.fit(X_train, y_train)
    pred_labels = clf.predict(X_dev)

    elapsed = time.time() - start

    return {
        "name": "bow_logistic_regression_mcq_only",
        "vectorizer": vectorizer,
        "model": clf,
        "pred_labels": pred_labels,
        "elapsed_sec": elapsed,
    }

bow_lr_run = train_bow_logistic_regression(mcq_train_df, mcq_dev_df)
bow_lr_scored, bow_lr_summary = evaluate_mcq_predictions(
    name=bow_lr_run["name"],
    dev_df=mcq_dev_df,
    pred_labels=bow_lr_run["pred_labels"],
    train_time_sec=bow_lr_run["elapsed_sec"],
)

Training BoW + Logistic Regression...
bow_logistic_regression_mcq_only
MCQ accuracy: 9 / 75 (12.00%)
Train time: 282.68 sec


## BoW + MLP

In [36]:
from sklearn.preprocessing import LabelEncoder

def train_bow_mlp(train_df: pd.DataFrame, dev_df: pd.DataFrame, max_features: int = 5000):
    start = time.time()

    vectorizer = CountVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b\w+\b",
        ngram_range=(1, 1),
        max_features=max_features,
        min_df=1,
        binary=True,
    )

    X_train = vectorizer.fit_transform(train_df["text"]).astype(np.float32).toarray()
    X_dev = vectorizer.transform(dev_df["text"]).astype(np.float32).toarray()

    # Encode labels A/B/C/D/... into integers to avoid sklearn string-label crash
    label_encoder = LabelEncoder()
    y_train_str = train_df["label"].astype(str).str.strip().str.upper().values
    y_train = label_encoder.fit_transform(y_train_str)

    clf = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        learning_rate_init=1e-3,
        batch_size=32,
        max_iter=100,
        early_stopping=False,   # IMPORTANT: avoid sklearn crash with string validation labels
        n_iter_no_change=10,
        random_state=RANDOM_SEED,
        verbose=True,
    )

    print("Training BoW + MLP...")
    clf.fit(X_train, y_train)

    pred_int = clf.predict(X_dev)
    pred_labels = label_encoder.inverse_transform(pred_int)

    elapsed = time.time() - start

    return {
        "name": "bow_mlp_mcq_only",
        "vectorizer": vectorizer,
        "label_encoder": label_encoder,
        "model": clf,
        "pred_labels": pred_labels,
        "elapsed_sec": elapsed,
    }


bow_mlp_run = train_bow_mlp(mcq_train_df, mcq_dev_df)

bow_mlp_scored, bow_mlp_summary = evaluate_mcq_predictions(
    name=bow_mlp_run["name"],
    dev_df=mcq_dev_df,
    pred_labels=bow_mlp_run["pred_labels"],
    train_time_sec=bow_mlp_run["elapsed_sec"],
)

Training BoW + MLP...
Iteration 1, loss = 2.32411354
Iteration 2, loss = 2.12425449
Iteration 3, loss = 1.92832732
Iteration 4, loss = 1.68582924
Iteration 5, loss = 1.39961334
Iteration 6, loss = 1.10096338
Iteration 7, loss = 0.82972729
Iteration 8, loss = 0.60765486
Iteration 9, loss = 0.45143139
Iteration 10, loss = 0.34734370
Iteration 11, loss = 0.27478911
Iteration 12, loss = 0.20962588
Iteration 13, loss = 0.17315718
Iteration 14, loss = 0.14368394
Iteration 15, loss = 0.11957326
Iteration 16, loss = 0.10149360
Iteration 17, loss = 0.08865884
Iteration 18, loss = 0.07883833
Iteration 19, loss = 0.06567143
Iteration 20, loss = 0.05966821
Iteration 21, loss = 0.06038492
Iteration 22, loss = 0.04946854
Iteration 23, loss = 0.05168120
Iteration 24, loss = 0.04082110
Iteration 25, loss = 0.03633501
Iteration 26, loss = 0.03383273
Iteration 27, loss = 0.03028014
Iteration 28, loss = 0.03142154
Iteration 29, loss = 0.02442572
Iteration 30, loss = 0.02721489
Iteration 31, loss = 0.0224

## Summary table

In [37]:
summary_df = pd.DataFrame([
    majority_summary,
    bow_lr_summary,
    bow_mlp_summary,
])

summary_path = RESULTS_DIR / "mcq_only_classical_baseline_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved summary to:", summary_path)
summary_df

Saved summary to: results/classical_baselines/mcq_only_classical_baseline_summary.csv


,run_name,task_scope,num_dev,num_correct,mcq_acc,free_form_acc,overall_acc,train_time_sec
0,majority_label_baseline_mcq_only,MCQ only,75,10,13.333333,N/A,N/A,0.000717
1,bow_logistic_regression_mcq_only,MCQ only,75,9,12.000000,N/A,N/A,282.676255
2,bow_mlp_mcq_only,MCQ only,75,8,10.666667,N/A,N/A,82.931287


## Save per-example predictions

In [38]:
bow_lr_scored.to_json(
    RESULTS_DIR / "bow_logistic_regression_mcq_dev_results.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

bow_mlp_scored.to_json(
    RESULTS_DIR / "bow_mlp_mcq_dev_results.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

majority_scored.to_json(
    RESULTS_DIR / "majority_label_mcq_dev_results.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Saved per-example results to:", RESULTS_DIR)

Saved per-example results to: results/classical_baselines


## Show failures

In [39]:
def show_failures(scored_df: pd.DataFrame, n: int = 10):
    wrong = scored_df[scored_df["correct"] == False].head(n)

    for _, row in wrong.iterrows():
        print("=" * 100)
        print("ID:", row["id"])
        print("Gold:", row["gold"])
        print("Pred:", row["pred_label"])
        print("Response:", row["response"])
        print("Question preview:")
        print(row["text"][:1000])

print("BoW Logistic Regression failures:")
show_failures(bow_lr_scored, n=5)

print("\n\nBoW MLP failures:")
show_failures(bow_mlp_scored, n=5)

BoW Logistic Regression failures:
ID: 875
Gold: H
Pred: E
Response: \boxed{E}
Question preview:
Let the function $y=f \left(x \right)$ satisfy the differential equation $y \prime+y={\frac{\mathrm{e}^{-x} \operatorname{c o s} x} {2 \sqrt{\operatorname{s i n} x}}}$, and $f \left(\pi\right)=0$, find the curve $y=f \left(x \right) \left(x \geqslant0 \right)$ The volume of the body of rotation obtained by rotating about the $x$ axis once is ().

Options:
A. $$ \frac{5\pi} {3 ( 5-\mathrm{e}^{-2\pi} )} $$
B. $$ \frac{2\pi} {5 ( 2-\mathrm{e}^{-\pi} )} $$
C. $$ \frac{2\pi} {5 ( 1-\mathrm{e}^{-\pi} )} $$
D. $$ \frac{3\pi} {3 ( 4-\mathrm{e}^{-2\pi} )} $$
E. $$ \frac{5\pi} {3 ( 5-\mathrm{e}^{-\pi} )} $$
F. $$ \frac{3\pi} {3 ( 5-\mathrm{e}^{-\pi} )} $$
G. $$ \frac{2\pi} {5 ( 2-\mathrm{e}^{-2\pi} )} $$
H. $$ \frac{\pi} {5 ( 1-\mathrm{e}^{-2 \pi} )} $$
I. $$ \frac{3\pi} {5 ( 2-\mathrm{e}^{-\pi} )} $$
J. $$ \frac{\pi} {5 ( 2-\mathrm{e}^{-\pi} )} $$
ID: 1116
Gold: A
Pred: B
Response: \boxed{B}
Question